# Model inference

This example demonstrates how to call models that are already deployed in the core gateway pattern.

Models live in three Foundry accounts behind a single APIM gateway. Teams never call
accounts directly - all traffic flows through the gateway using a team-scoped APIM
subscription key. Each team has a Foundry project with a named connection to the gateway.

| Access path | Use case |
|---|---|
| Direct APIM client (`AzureOpenAI`) | Chat completions, embeddings, research model |
| Project client (`AIProjectClient`) | Responses API, multi-turn, streaming - Foundry-native surface |

> **Auth note:** Key-based access is disabled on hub accounts by Azure policy. The APIM
> gateway authenticates to Foundry core accounts using managed identity. Teams authenticate to
> the gateway using their APIM subscription key (`{TEAM}_GATEWAY_KEY`).

## Prerequisites

1. **Python environment**: Run `uv sync` from the repository root, then select the `.venv`
   kernel in VS Code.
2. **`.env` file**: Must be populated by the `05-foundry-project-pattern-setup` labs:
   - `GATEWAY_URL` - APIM gateway URL (set by the core gateway deployment)
   - `ALPHA_GATEWAY_KEY` - Team Alpha APIM subscription key (set by the core gateway deployment)
   - `CHAT_MODEL`, `EMBEDDING_MODEL`, `RESEARCH_MODEL` - model deployment names (set by the core gateway deployment)
   - `ALPHA_FOUNDRY_PROJECT_ENDPOINT` - Team Alpha project endpoint (set by the project spoke deployment)
   - `ALPHA_FOUNDRY_CORE_CONNECTION` - Team Alpha APIM connection name, e.g. `core-alpha` (set by the project spoke deployment)
3. **Azure CLI**: Run `az login` so `DefaultAzureCredential` resolves for the project client.

## Imports and configuration

Load `.env` from the repository root. All model names and endpoints are read from environment
variables - no hard-coded values in this notebook.

In [1]:
import os
import subprocess
from pathlib import Path
from dotenv import load_dotenv

repo_root = Path(subprocess.run(
    'git rev-parse --show-toplevel', shell=True, capture_output=True, text=True
).stdout.strip())
load_dotenv(repo_root / '.env', override=True)

# Team Alpha project - 1:1 spoke pattern
PROJECT_ENDPOINT = os.environ["ALPHA_FOUNDRY_PROJECT_ENDPOINT"]
CORE_CONNECTION   = os.environ["ALPHA_FOUNDRY_CORE_CONNECTION"]  # e.g. "core-alpha"

# APIM gateway (shared by all teams, set by 04-02)
GATEWAY_URL      = os.environ["GATEWAY_URL"]        # https://apim-foundry-{suffix}.azure-api.net/openai
GATEWAY_KEY      = os.environ["ALPHA_GATEWAY_KEY"]  # Team Alpha subscription key
GATEWAY_ENDPOINT = GATEWAY_URL.removesuffix("/openai")  # base URL for AzureOpenAI client

# Model deployment names (hosted in hub accounts, served via APIM)
CHAT_MODEL     = os.environ["CHAT_MODEL"]                          # gpt-4.1-mini
EMBEDDING_MODEL = os.environ["EMBEDDING_MODEL"]                    # text-embedding-3-large
RESEARCH_MODEL  = os.environ["RESEARCH_MODEL"]                     # o3-deep-research

print(f"Project endpoint : {PROJECT_ENDPOINT}")
print(f"Hub connection   : {CORE_CONNECTION}")
print(f"Gateway endpoint : {GATEWAY_ENDPOINT}")
print(f"Chat model       : {CHAT_MODEL}")
print(f"Embedding model  : {EMBEDDING_MODEL}")
print(f"Research model   : {RESEARCH_MODEL}")


Project endpoint : https://aif-spoke-alpha-c2676f.services.ai.azure.com/api/projects/project-alpha-c2676f
Hub connection   : core-alpha
Gateway endpoint : https://apim-foundry-c2676f.azure-api.net
Chat model       : gpt-4.1-mini
Embedding model  : text-embedding-3-large
Research model   : o3-deep-research


## Clients

Two clients are used throughout this notebook:

- **`apim_client`** - `AzureOpenAI` pointed at the APIM gateway with an API key. Supports
  all Azure OpenAI API surfaces: `chat.completions`, `embeddings`, `responses`.
- **`project_client`** - `AIProjectClient` pointed at the Team Alpha Foundry project. Uses
  `DefaultAzureCredential` for auth. Provides the Foundry-native `get_openai_client()` which
  routes through the project's APIM connection (`core-alpha`). Models are referenced as
  `{connection}/{model}` (e.g. `core-alpha/gpt-4.1-mini`) when calling the Responses API.

## Access patterns

Two inference paths are available within the core gateway pattern. Both ultimately reach the
same APIM gateway and the same hub Foundry accounts - the difference is the SDK layer and
which API surfaces each path supports.

### Direct APIM client

```python
apim_client = AzureOpenAI(azure_endpoint=GATEWAY_ENDPOINT, api_key=GATEWAY_KEY)
```

The team calls the APIM gateway directly as an Azure OpenAI-compatible endpoint, authenticated
with their APIM subscription key. The full Azure OpenAI API surface is available.

### Foundry project client

```python
project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
with project_client.get_openai_client() as client:
    client.responses.create(model="core-alpha/gpt-4.1-mini", ...)
```

The team calls through their Foundry project, which routes the request via the project's
registered APIM connection (`core-alpha`). The project holds the APIM key internally - the
client authenticates to the project using `DefaultAzureCredential`. Models must be referenced
as `{connection}/{model}` (e.g. `core-alpha/gpt-4.1-mini`).

The Foundry project connection type (`ApiManagement`) only supports the **Responses API**
path. `chat.completions` and `embeddings` are not supported through this connection type.

### What works where

| API surface | Direct APIM client | Foundry project client |
|---|---|---|
| `chat.completions` | yes | no |
| `embeddings` | yes | no |
| `responses` (Responses API) | yes | yes |
| `previous_response_id` (multi-turn) | yes | yes |
| `stream=True` | yes | yes |

**Rule of thumb:** use the project client for anything Foundry-native (agents, multi-turn
conversation tracking, streaming via the Responses API). Use the direct APIM client for
`chat.completions`, embeddings, and the model router.

In [2]:
from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

# Direct APIM client - chat completions, embeddings, and all standard Azure OpenAI surfaces
apim_client = AzureOpenAI(
    azure_endpoint=GATEWAY_ENDPOINT,
    api_key=GATEWAY_KEY,
    api_version="2024-10-21",
)

# Foundry project client - Responses API, agents, connections
credential     = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)

print("apim_client    : ready")
print("project_client : ready")

apim_client    : ready
project_client : ready


## Why APIM connections require the Responses API

Foundry project connections of type `ApiManagement` only expose the Responses API path.
Calling `chat.completions` through the project client raises an error because the APIM
policy does not forward that surface through the connection.

The working pattern is to create an agent with a `PromptAgentDefinition` (which stores the
model and system instructions in the project) and then invoke it via `responses.create` using
an `agent_reference`. The Responses API path is what APIM connections support.

The cell below demonstrates both cases explicitly.

In [3]:
from azure.ai.projects.models import PromptAgentDefinition

hub_model  = f"{CORE_CONNECTION}/{CHAT_MODEL}"   # e.g. "core-alpha/gpt-4.1-mini"
openai_client = project_client.get_openai_client()

print(f"Project  : {PROJECT_ENDPOINT}")
print(f"Model    : {hub_model}")
print()

# ❌ chat.completions - not supported through an ApiManagement project connection
print("❌ chat.completions.create():")
try:
    resp = openai_client.chat.completions.create(
        model=hub_model,
        messages=[{"role": "user", "content": "Name a planet with rings."}],
    )
    print(f"   {resp.choices[0].message.content[:80]}")
except Exception as e:
    print(f"   Error: {str(e)[:120]}")

print()

# ✅ Agent + Responses API - the supported path for ApiManagement connections
print("✅ Agent + responses.create():")
agent = project_client.agents.create_version(
    agent_name="apim-demo-agent",
    definition=PromptAgentDefinition(
        model=hub_model,
        instructions="You are a space exploration expert. Answer in one sentence.",
    ),
)
resp = openai_client.responses.create(
    input="Name a planet with rings.",
    extra_body={
        "agent_reference": {
            "name": agent.name,
            "version": agent.version,
            "type": "agent_reference",
        }
    },
)
print(f"   {resp.output_text}")

# Clean up demo agent
project_client.agents.delete(agent_name=agent.name)
openai_client.close()

Project  : https://aif-spoke-alpha-c2676f.services.ai.azure.com/api/projects/project-alpha-c2676f
Model    : core-alpha/gpt-4.1-mini

❌ chat.completions.create():
   Error: Error code: 404 - {'error': {'type': 'invalid_request_error', 'code': 'DeploymentNotFound', 'message': 'The API deployme

✅ Agent + responses.create():
   Saturn is a planet with prominent rings.


## Agent API across team projects

The Agent API is the primary inference pattern for Foundry projects that access models via
APIM connections. The key convention is the `{connection}/{model}` model reference format -
the connection name tells Foundry which APIM connection to route through, and the model name
identifies the deployment on the other side of the gateway.

```
core-alpha/gpt-4.1-mini      ← Team Alpha (1:1 spoke)
core-beta/gpt-4.1-mini       ← Team Beta  (1:N multi-project)
core-delta/gpt-4.1-mini      ← Team Delta
core-gamma/gpt-4.1-mini      ← Team Gamma
```

The cell below creates a short-lived agent in each team project, queries it, and cleans up.
Teams that have not been deployed (missing env vars) are skipped automatically.

In [4]:
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition

# All team projects - teams with missing env vars are skipped
teams = [
    {
        "display":     "Team Alpha (1:1 spoke)",
        "endpoint":    os.environ.get("ALPHA_FOUNDRY_PROJECT_ENDPOINT"),
        "connection":  os.environ.get("ALPHA_FOUNDRY_CORE_CONNECTION"),
    },
    {
        "display":     "Team Beta (1:N multi-project)",
        "endpoint":    os.environ.get("BETA_FOUNDRY_PROJECT_ENDPOINT"),
        "connection":  os.environ.get("BETA_FOUNDRY_CORE_CONNECTION"),
    },
    {
        "display":     "Team Delta (1:N multi-project)",
        "endpoint":    os.environ.get("DELTA_FOUNDRY_PROJECT_ENDPOINT"),
        "connection":  os.environ.get("DELTA_FOUNDRY_CORE_CONNECTION"),
    },
    {
        "display":     "Team Gamma (1:N multi-project)",
        "endpoint":    os.environ.get("GAMMA_FOUNDRY_PROJECT_ENDPOINT"),
        "connection":  os.environ.get("GAMMA_FOUNDRY_CORE_CONNECTION"),
    },
]

credential = DefaultAzureCredential()
AGENT_NAME = "space-exploration-agent"
QUERY      = "What is the largest planet in our solar system?"

for team in teams:
    if not team["endpoint"] or not team["connection"]:
        print(f"  ⚠  {team['display']} - skipped (env vars not set)")
        continue

    gateway_model = f"{team['connection']}/{CHAT_MODEL}"
    print(f"\n{team['display']}")
    print(f"  endpoint      : {team['endpoint']}")
    print(f"  gateway model : {gateway_model}")

    client       = AIProjectClient(credential=credential, endpoint=team["endpoint"])
    openai_client = client.get_openai_client()

    try:
        agent = client.agents.create_version(
            agent_name=AGENT_NAME,
            definition=PromptAgentDefinition(
                model=gateway_model,
                instructions="You are a space exploration expert. Answer in one sentence.",
            ),
        )

        resp = openai_client.responses.create(
            input=QUERY,
            extra_body={
                "agent_reference": {
                    "name":    agent.name,
                    "version": agent.version,
                    "type":    "agent_reference",
                }
            },
        )

        print(f"  response      : {resp.output_text}")
    except Exception as e:
        print(f"  error         : {str(e)[:120]}")
    finally:
        client.agents.delete(agent_name=AGENT_NAME)
        openai_client.close()


Team Alpha (1:1 spoke)
  endpoint      : https://aif-spoke-alpha-c2676f.services.ai.azure.com/api/projects/project-alpha-c2676f
  gateway model : core-alpha/gpt-4.1-mini
  response      : The largest planet in our solar system is Jupiter.

Team Beta (1:N multi-project)
  endpoint      : https://aif-spoke-multi-c2676f.services.ai.azure.com/api/projects/project-beta-c2676f
  gateway model : core-beta/gpt-4.1-mini
  error         : Error code: 400 - {'error': {'message': "Connection 'core-beta' not found.", 'type': 'invalid_request_error', 'param': N

Team Delta (1:N multi-project)
  endpoint      : https://aif-spoke-multi-c2676f.services.ai.azure.com/api/projects/project-delta-c2676f
  gateway model : core-delta/gpt-4.1-mini
  error         : Error code: 400 - {'error': {'message': "Connection 'core-delta' not found.", 'type': 'invalid_request_error', 'param': 

Team Gamma (1:N multi-project)
  endpoint      : https://aif-spoke-multi-c2676f.services.ai.azure.com/api/projects/project-gam

## Chat

Standard chat completions via the APIM gateway. APIM routes the request to `aif-core` using
managed identity - the API key is only used between the client and APIM, never beyond it.

This is the same surface any Azure OpenAI client uses; the only difference is the endpoint
is the APIM gateway rather than a Foundry account directly.

In [5]:
response = apim_client.chat.completions.create(
    model=CHAT_MODEL,
    messages=[
        {"role": "system", "content": "You are a concise technical assistant."},
        {"role": "user",   "content": "What is catastrophic forgetting in neural networks?"},
    ],
)

print(f"Model  : {response.model}")
print(f"Tokens : {response.usage.total_tokens}")
print()
print(response.choices[0].message.content)

Model  : gpt-4.1-mini-2025-04-14
Tokens : 100

Catastrophic forgetting in neural networks refers to the issue where a model trained sequentially on multiple tasks rapidly loses performance on previously learned tasks when it is trained on new tasks. Essentially, the network "forgets" old knowledge as it overwrites its parameters to fit the new data, making it difficult to retain information over time without explicit mechanisms to preserve earlier learning.


## Embedding model

`text-embedding-3-large` is deployed on `aif-core`. APIM's default routing rule forwards all
requests that are not matched by a specific rule (research, OSS) to `aif-core`, so embeddings
reach the right backend automatically.

In [6]:
texts = [
    "Azure AI Foundry centralises model governance behind a single gateway.",
    "APIM routes model requests to the correct regional backend using managed identity.",
    "Each team project has its own APIM subscription key for independent rate limiting.",
]

result = apim_client.embeddings.create(model=EMBEDDING_MODEL, input=texts)

print(f"Model      : {EMBEDDING_MODEL}")
print(f"Dimensions : {len(result.data[0].embedding)}")
print()
for i, item in enumerate(result.data):
    vec = item.embedding
    print(f"[{i}] [{vec[0]:.6f}, {vec[1]:.6f}, {vec[2]:.6f}, ...]  ({len(vec)} dims)")

Model      : text-embedding-3-large
Dimensions : 3072

[0] [-0.018051, 0.022552, -0.020172, ...]  (3072 dims)
[1] [-0.030518, -0.005497, -0.018631, ...]  (3072 dims)
[2] [-0.052673, -0.005455, -0.015823, ...]  (3072 dims)


## Research model

`o3-deep-research` is deployed on `aif-research` in Norway East. APIM matches the URL pattern
`/deployments/o3-deep-research/*` and forwards the request to the research hub backend.
From the client's perspective the endpoint is the same gateway URL - region routing is
entirely handled by APIM policy.

This model requires `api_version="2024-12-01-preview"` and may take longer to respond.

In [7]:
research_client = AzureOpenAI(
    azure_endpoint=GATEWAY_ENDPOINT,
    api_key=GATEWAY_KEY,
    api_version="2024-12-01-preview",
    timeout=120,
)

response = research_client.chat.completions.create(
    model=RESEARCH_MODEL,
    messages=[
        {
            "role": "user",
            "content": (
                "Briefly summarise the key trade-offs between transformer and "
                "state-space sequence models such as Mamba."
            ),
        }
    ],
)

print(f"Model  : {response.model}")
print(f"Tokens : {response.usage.total_tokens}")
print()
print(response.choices[0].message.content)

Model  : o3-deep-research
Tokens : 9848

**Transformers vs. State-Space Models (e.g., Mamba) – Key Trade-offs:**

- ****Representation & Flexibility**: **Transformers** use **self-attention**, which allows them to flexibly model relationships between any pair of positions in a sequence. This gives them great power to capture complex, long-range dependencies **dynamically** (i.e., deciding which tokens to attend to based on content). In contrast, **state-space models (SSMs)** like **Mamba** use **learned linear recurrences / state transitions** (often derived from continuous-time state-space equations) to process sequences. These models come with **strong inductive biases** for sequential data (like built-in notions of continuity or decay over time), which can be advantageous for tasks with very long sequences or signals. However, this structure may make them **less universally flexible** than transformers for capturing arbitrary pairwise interactions, since their “attention” is essenti

## Responses API

The Responses API is the Foundry-native inference surface. It is accessed through the project
client rather than the direct APIM client. The project client routes the request through the
project's registered APIM connection (`core-alpha`), so the model is referenced as
`{connection}/{model}` - for example `core-alpha/gpt-4.1-mini`.

Choose the Responses API when:
- You need server-side conversation tracking via `previous_response_id`
- You are building agents that run on the Foundry Agent Service
- You prefer a simpler `input=` parameter over constructing a full `messages[]` array

In [8]:
hub_model = f"{CORE_CONNECTION}/{CHAT_MODEL}"  # e.g. "core-alpha/gpt-4.1-mini"

with project_client.get_openai_client() as client:
    response = client.responses.create(
        model=hub_model,
        input="Define catastrophic forgetting.",
    )

print(f"Model          : {hub_model}")
print(f"Response ID    : {response.id}")
print()
print(response.output_text)

Model          : core-alpha/gpt-4.1-mini
Response ID    : resp_08cb11ad7fb94425006a00778c580c8194a03a7c4c5a3c6ff6

**Catastrophic forgetting** (also known as catastrophic interference) is a phenomenon in machine learning, particularly in neural networks, where a model that is trained sequentially on multiple tasks or datasets tends to forget previously learned information when it learns new tasks. This results in a significant and abrupt loss of performance on earlier tasks after training on new ones, making it difficult for the model to retain knowledge over time when learning incrementally.


## Multi-turn

Pass `previous_response_id` to chain calls server-side. The model receives the full
conversation history without the client re-sending it - Foundry stores the prior response
and appends the new turn automatically.

> **Billing note:** All prior input tokens are charged on every subsequent call. There is
> no server-side token deduplication. Cost is equivalent to sending the full `messages[]`
> array on each turn.

In [9]:
with project_client.get_openai_client() as client:
    first = client.responses.create(
        model=hub_model,
        input="Define catastrophic forgetting.",
    )
    second = client.responses.create(
        model=hub_model,
        previous_response_id=first.id,
        input=[{"role": "user", "content": "Explain it to a 10-year-old."}],
    )

print("--- Turn 1 ---")
print(first.output_text)
print()
print("--- Turn 2 (chained via previous_response_id) ---")
print(second.output_text)

--- Turn 1 ---
Catastrophic forgetting, also known as catastrophic interference, is a phenomenon in machine learning and neural networks where a model abruptly and significantly loses previously acquired knowledge upon learning new information. This typically occurs in sequential or continual learning settings, where the model is trained on multiple tasks or datasets one after another. Instead of retaining earlier knowledge while learning new tasks, the model’s performance on earlier tasks degrades drastically.

In other words, when a neural network is trained on a new task, its parameters adjust to optimize performance on that task, often overwriting the parameter configurations that were important for earlier tasks, leading to forgetting of previously learned information. This is a major challenge for developing AI systems that can learn continuously over time without forgetting prior skills or knowledge.

--- Turn 2 (chained via previous_response_id) ---
Sure! Imagine you have a box

## Streaming

Pass `stream=True` to receive incremental tokens as the model generates them. Key event types:

- `response.output_text.delta` - incremental text token; read `event.delta`
- `response.output_text.done` - signals the text output is complete

In [10]:
with project_client.get_openai_client() as client:
    stream = client.responses.create(
        model=hub_model,
        input="List three real-world applications of reinforcement learning.",
        stream=True,
    )
    for event in stream:
        if event.type == "response.output_text.delta":
            print(event.delta, end="", flush=True)
print()

Here are three real-world applications of reinforcement learning (RL):

1. **Autonomous Vehicles:**  
   RL is used to train self-driving cars to make decisions in complex and dynamic environments, such as navigating traffic, controlling speed, and avoiding collisions.

2. **Robotics:**  
   Robots can learn tasks like grasping objects, walking, or assembling parts through RL, improving their adaptability and performance without explicit programming for every scenario.

3. **Recommendation Systems:**  
   RL helps personalize recommendations (e.g., for movies, products, or content) by learning users’ preferences over time and optimizing for long-term engagement.

If you'd like, I can provide more examples or elaborate on any of these!
